# v3a — ConvNeXt Multi-Scale + Custom Transformer Decoder

| Field | Value |
|---|---|
| Framework | PyTorch |
| Dataset | Flickr30k (31,783 images, 5 captions each) |
| Image Encoder | ConvNeXt-Tiny multi-scale — stages 1,2,3 → (B, 64, 768) |
| Projection | ProjectionHead: Linear(768,512) → GELU → Linear(512,512) → LayerNorm |
| Decoder | Custom TransformerDecoder — 6 layers, 8 heads, 512-dim + sinusoidal PE |
| Tokenizer | GPT-2 BPE (vocab 50,257; pad = eos = 50256) |
| Feature Storage | LMDB — pre-computed ConvNeXt embeddings (float16, 64×768) |
| Training | Adam lr=1e-5, CrossEntropyLoss, 10 epochs |
| Inference | Top-k=50 + multinomial sampling |
| BLEU-4 | 0.0674 |
| Status | Baseline; superseded by v3b (Perceiver + GPT-2 cross-attention) |

## Overview

This is Phase 1 of the transformer-based rewrite. After v2 hit a ceiling of BLEU 0.026 with a TF/Keras attention model on Instagram captions, the architecture was rethought from scratch in PyTorch.

Two key changes from v2:
1. **InceptionV3 → ConvNeXt-Tiny multi-scale.** Instead of a single flat feature vector or 64 spatial patches from one layer, we extract from three stages of ConvNeXt and concatenate them — giving the decoder richer, hierarchical visual tokens.
2. **LSTM → custom TransformerDecoder.** Replaces the recurrent decoder with a 6-layer cross-attention Transformer, enabling the model to attend to all image tokens at every decoding step.

Because ConvNeXt feature extraction is expensive at training time, embeddings are pre-computed for all images and stored in LMDB (Lightning Memory-Mapped Database). Training reads from LMDB directly — no GPU time spent on the image encoder during the training loop.

BLEU-4 improved from 0.026 (v2) to **0.0674** (v3a). The model generates fluent sentences but struggles with visual grounding — a limitation explored in v3b.

## 1. Setup

In [ ]:
import os
import re
import csv
import math
import random
import hashlib
import numpy as np
import lmdb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from torchvision import transforms
from PIL import Image

from transformers import GPT2Tokenizer

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download('punkt', quiet=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Dataset — Flickr30k

Flickr30k contains 31,783 images with 5 human-written captions each. Compared to the Instagram dataset used in v1/v2, Flickr30k captions are:
- Descriptive rather than social ("A man in a red shirt is riding a bicycle" vs "Living my best life")
- Grammatically consistent — better signal for a language model to learn from
- Multi-reference — having 5 references per image makes BLEU evaluation more reliable

Captions are loaded from `captions.txt`, a CSV with columns `image_name`, `comment_number`, `comment`.

In [ ]:
def load_captions(filepath):
    captions_dict = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='|')
        for row in reader:
            image_name = row['image_name'].strip()
            comment_num = row[' comment_number'].strip()
            comment = row[' comment'].strip()
            if image_name not in captions_dict:
                captions_dict[image_name] = {}
            captions_dict[image_name][comment_num] = comment
    return captions_dict

FLICKR_PATH = '/kaggle/input/flickr30k/captions.txt'
flickr_captions_dict = load_captions(FLICKR_PATH)

all_keys = list(flickr_captions_dict.keys())
print(f'Total images: {len(all_keys)}')
print(f'Sample key: {all_keys[0]}')
print(f'Sample captions: {flickr_captions_dict[all_keys[0]]}')

In [ ]:
random.seed(42)
random.shuffle(all_keys)

split = int(0.9 * len(all_keys))
train_keys = all_keys[:split]
val_keys   = all_keys[split:]

print(f'Train: {len(train_keys)} images ({len(train_keys)*5} caption pairs)')
print(f'Val:   {len(val_keys)} images ({len(val_keys)*5} caption pairs)')

## 3. Tokenizer — GPT-2 BPE

GPT-2's BPE tokenizer is used throughout Phase 3. Key properties:
- Fixed vocabulary of 50,257 tokens — no OOV problem
- Subword tokenization handles rare and compound words
- The EOS token (50256) doubles as BOS and PAD — GPT-2 has no dedicated PAD token

The decoder uses `CrossEntropyLoss(ignore_index=50256)` to exclude the padding token from the loss.

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token  # 50256

PAD_ID = tokenizer.pad_token_id   # 50256
BOS_ID = tokenizer.bos_token_id   # 50256
EOS_ID = tokenizer.eos_token_id   # 50256

print(f'Vocab size: {tokenizer.vocab_size}')
print(f'PAD/BOS/EOS token id: {PAD_ID}')

sample = tokenizer.encode('A man in a red shirt is riding a bicycle.')
print(f'Sample encoding: {sample}')
print(f'Decoded back: {tokenizer.decode(sample)}')

## 4. Image Encoder — ConvNeXtMultiScale

The core upgrade from v2: instead of one feature layer, we pull from three stages of ConvNeXt-Tiny.

**Why multi-scale?**
- Stage 1 (192-dim): fine-grained textures and edges
- Stage 2 (384-dim): mid-level patterns (object parts, shapes)
- Stage 3 (768-dim): high-level semantics (objects, scenes)

Each stage is spatially pooled to 8×8, flattened to 64 tokens, projected to 256-dim, then concatenated channel-wise to give (B, 64, 768) — 64 spatial positions each carrying features from all three scales simultaneously.

The encoder is **frozen** throughout training. Feature extraction is done once offline and stored in LMDB.

In [ ]:
class ConvNextMultiScale(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = timm.create_model(
            'convnext_tiny.fb_in22k_ft_in1k',
            pretrained=True,
            features_only=True,
            out_indices=(1, 2, 3)        # stages with channels 192, 384, 768
        )
        for param in self.model.parameters():
            param.requires_grad = False  # frozen

        self.pool = nn.AdaptiveAvgPool2d((8, 8))
        self.projections = nn.ModuleList([
            nn.Linear(192, 256),
            nn.Linear(384, 256),
            nn.Linear(768, 256),
        ])
        self.norm = nn.LayerNorm(768)    # normalizes the concatenated (B,64,768)

    def forward(self, img):
        stage_outputs = self.model(img)  # list of 3 feature maps
        tokens = []
        for feat, proj in zip(stage_outputs, self.projections):
            x = self.pool(feat)           # (B, C, 8, 8)
            x = x.flatten(2).transpose(1, 2)  # (B, 64, C)
            x = proj(x)                   # (B, 64, 256)
            tokens.append(x)
        out = torch.cat(tokens, dim=2)   # (B, 64, 768)
        out = self.norm(out)
        return out

## 5. Feature Pre-computation — LMDB

Running ConvNeXt on every image at every training epoch is expensive. Instead, embeddings are extracted once and stored in LMDB — a file-based key-value store that supports fast random access via memory-mapped reads.

Each key is an image filename. Each value is the (64, 768) embedding stored as `float16` (halves memory, negligible precision loss for downstream training).

For Flickr30k (31,783 images), the LMDB is sharded across 6 parts (`lmdb_part_1` through `lmdb_part_6`) to stay within Kaggle's dataset size limits. `get_embedding()` tries all shards sequentially.

The extraction loop below is shown for reference — it was run once offline, not during training.

In [ ]:
# --- Feature extraction (run once offline) ---
# encoder = ConvNextMultiScale().to(device).eval()
#
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225]),
# ])
#
# env = lmdb.open('/kaggle/working/flickr_lmdb', map_size=10*1024**3)
# with torch.no_grad():
#     for image_name in all_keys:
#         img = Image.open(os.path.join(IMAGE_DIR, image_name)).convert('RGB')
#         img_tensor = transform(img).unsqueeze(0).to(device)
#         embedding = encoder(img_tensor).squeeze(0).cpu().numpy().astype(np.float16)
#         with env.begin(write=True) as txn:
#             txn.put(image_name.encode(), embedding.tobytes())

In [ ]:
import lmdb

N_SHARDS = 6
envs = [
    lmdb.open(
        f'/kaggle/input/datasets/huzefamerchant/lmdb-part-{i}',
        readonly=True, lock=False, readahead=False
    )
    for i in range(1, N_SHARDS + 1)
]

def get_embedding(key):
    key_bytes = key.encode()
    for env in envs:
        with env.begin() as txn:
            value = txn.get(key_bytes)
            if value is not None:
                return np.frombuffer(value, dtype=np.float16).reshape(64, 768)
    return None

# Sanity check
sample_emb = get_embedding(all_keys[0])
print(f'Embedding shape: {sample_emb.shape}')   # (64, 768)
print(f'Embedding dtype: {sample_emb.dtype}')   # float16

## 6. Dataset and DataLoader

Each training sample is one (image, caption) pair. Since each image has 5 captions, the effective dataset size is `num_images × 5`.

`__getitem__` loads the pre-computed LMDB embedding and tokenizes one caption. The caption is wrapped with BOS/EOS tokens and padded or truncated to `max_len`.

In [ ]:
MAX_LEN = 64

class FlickrDataset(Dataset):
    def __init__(self, keys, captions_dict, tokenizer, max_len=MAX_LEN):
        self.samples = []
        for key in keys:
            for caption in captions_dict[key].values():
                self.samples.append((key, caption))
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        key, caption = self.samples[idx]

        embedding = get_embedding(key)
        image_tensor = torch.from_numpy(embedding.copy()).float()   # (64, 768)

        token_ids = self.tokenizer.encode(caption)
        token_ids = [BOS_ID] + token_ids + [EOS_ID]
        token_ids = token_ids[:self.max_len]
        token_ids += [PAD_ID] * (self.max_len - len(token_ids))

        caption_tensor = torch.tensor(token_ids, dtype=torch.long)
        return image_tensor, caption_tensor


train_dataset = FlickrDataset(train_keys, flickr_captions_dict, tokenizer)
val_dataset   = FlickrDataset(val_keys,   flickr_captions_dict, tokenizer)

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}')

img, cap = train_dataset[0]
print(f'Image tensor shape: {img.shape}')     # (64, 768)
print(f'Caption tensor shape: {cap.shape}')   # (64,)

## 7. Projection Head

The LMDB embeddings are (64, 768). The Transformer decoder operates in 512-dim. The ProjectionHead bridges this gap:

```
(B, 64, 768) → Linear(768→512) → GELU → Linear(512→512) → LayerNorm → (B, 64, 512)
```

This is the only trainable image-side component in Phase 1 — the ConvNeXt encoder itself is frozen.

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1   = nn.Linear(768, 512)
        self.gelu      = nn.GELU()
        self.linear2   = nn.Linear(512, 512)
        self.layernorm = nn.LayerNorm(512)

    def forward(self, x):        # x: (B, 64, 768)
        x = self.linear1(x)      # (B, 64, 512)
        x = self.gelu(x)
        x = self.linear2(x)      # (B, 64, 512)
        x = self.layernorm(x)
        return x                 # (B, 64, 512)

## 8. Sinusoidal Positional Encoding

The custom Transformer does not use learned positional embeddings (unlike GPT-2). Fixed sinusoidal encoding is used instead:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right) \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

This is pre-computed once and registered as a buffer inside the Transformer — it moves to the correct device automatically and is not a learned parameter.

In [ ]:
def positionalencoding(max_len, d_model):
    position = torch.arange(max_len).unsqueeze(1)                          # (max_len, 1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
    )                                                                       # (d_model/2,)
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe                                                               # (max_len, d_model)

positional_encoding = positionalencoding(2000, 512)
print(f'Positional encoding shape: {positional_encoding.shape}')  # (2000, 512)

## 9. Custom Transformer Decoder

A 6-layer TransformerDecoder built from PyTorch primitives:

- **Token embedding:** `nn.Embedding(50257, 512)` — learned, 50,257 GPT-2 vocabulary tokens
- **Positional encoding:** sinusoidal (pre-computed, added to embeddings)
- **6× `TransformerDecoderLayer`:** 512-dim, 8 heads, GELU activation, `batch_first=True`
  - Each layer does: masked self-attention → cross-attention over image memory → FFN
- **Output projection:** `nn.Linear(512, 50257)` → logits over GPT-2 vocabulary

The causal mask (`generate_square_subsequent_mask`) prevents each position from attending to future tokens. `tgt_is_causal=True` enables the efficient PyTorch implementation.

The forward output is `(B, 50257, T)` — transposed for `nn.CrossEntropyLoss` which expects `(B, C, T)`.

In [ ]:
class Transformer(nn.Module):
    def __init__(self, positional_encoding):
        super().__init__()
        self.register_buffer('positional_encoding', positional_encoding)  # (2000, 512)
        self.embedding = nn.Embedding(50257, 512)
        self.decoder   = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=512, nhead=8, activation='gelu', batch_first=True
            )
            for _ in range(6)
        ])
        self.output_proj = nn.Linear(512, 50257)

    def forward(self, memory, caption_ids):   # memory: (B,64,512)  caption_ids: (B,T)
        T = caption_ids.shape[1]
        mask = nn.Transformer.generate_square_subsequent_mask(T).to(caption_ids.device)

        x = self.embedding(caption_ids)       # (B, T, 512)
        x = x + self.positional_encoding[:T].unsqueeze(0)   # add sinusoidal PE

        for layer in self.decoder:
            x = layer(x, memory=memory, tgt_mask=mask, tgt_is_causal=True)

        logits = self.output_proj(x)          # (B, T, 50257)
        return logits.transpose(1, 2)         # (B, 50257, T) — for CrossEntropyLoss

## 10. Training

Both `ProjectionHead` and `Transformer` are trained jointly. ConvNeXt stays frozen.

**Loss:** `nn.CrossEntropyLoss(ignore_index=50256)` — padding token 50256 is excluded from the loss computation.

**Optimizer:** Adam, lr=1e-3 for the projection head, lr=1e-5 for the decoder. (Single lr=1e-5 in early runs.)

**Training run:** 50 epochs total with early stopping (patience=7 on val loss, reload best weights). The `EPOCHS=10` variable in the loop below is the per-session cap; the actual training run went the full 50 epochs.

In [ ]:
import time

def train(dataloader, projection_head, decoder, loss_fn, optimizer):
    projection_head.train()
    decoder.train()
    size = len(dataloader.dataset)

    for batch, (train_image, train_caption) in enumerate(dataloader):
        train_image   = train_image.to(device)
        train_caption = train_caption.to(device)

        input_seq = train_caption[:, :-1]    # (B, T-1) — decoder input
        y         = train_caption[:, 1:]     # (B, T-1) — shifted targets

        memory = projection_head(train_image)  # (B, 64, 512)
        pred   = decoder(memory, input_seq)    # (B, 50257, T-1)
        loss   = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            current = (batch + 1) * len(train_image)
            print(f'loss: {loss.item():>7.4f}  [{current:>6d}/{size:>6d}]')


def evaluate(dataloader, projection_head, decoder, loss_fn):
    projection_head.eval()
    decoder.eval()
    total_loss = 0
    num_batches = len(dataloader)

    with torch.no_grad():
        for val_image, val_caption in dataloader:
            val_image   = val_image.to(device)
            val_caption = val_caption.to(device)
            input_seq   = val_caption[:, :-1]
            y           = val_caption[:, 1:]
            memory      = projection_head(val_image)
            pred        = decoder(memory, input_seq)
            total_loss += loss_fn(pred, y).item()

    avg_loss = total_loss / num_batches
    print(f'Val loss: {avg_loss:.4f}')
    return avg_loss

In [ ]:
EPOCHS  = 10   # per-session cap; actual training run went to 50 epochs
LR      = 1e-5
PATIENCE = 7

projection_head = ProjectionHead().to(device)
decoder         = Transformer(positional_encoding).to(device)

loss_fn   = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(
    list(projection_head.parameters()) + list(decoder.parameters()),
    lr=LR
)

best_val_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    start = time.perf_counter()
    print(f'Epoch {epoch+1}\n{"-"*50}')

    train(train_dataloader, projection_head, decoder, loss_fn, optimizer)
    val_loss = evaluate(val_dataloader, projection_head, decoder, loss_fn)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(projection_head.state_dict(), '/kaggle/working/best_projection_head.pth')
        torch.save(decoder.state_dict(),         '/kaggle/working/best_decoder.pth')
        print('  Saved best weights.')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}.')
            break

    elapsed = time.perf_counter() - start
    print(f'  Time: {elapsed:.1f}s\n')

print('Training complete.')

## 11. Inference

`caption_generator` uses **top-k=50 + multinomial sampling** (temperature=1.0 in Phase 1, temperature=0.7 from Phase 2d onwards).

Decoding loop:
1. Start sequence: `[BOS]` (token 50256)
2. Feed sequence through `ProjectionHead → Transformer`
3. Extract logits at the last position → take top-50 → softmax → sample one token
4. Append token; stop if: EOS (50256) reached, full-stop `.` (token 764) reached, or 3 consecutive repeated tokens
5. Decode with GPT-2 tokenizer

No beam search in Phase 1 (that's added from Phase 2d onward). v3b reduces k from 50 to 20 because GPT-2 produces sharper token distributions than a randomly-initialized decoder — a wide pool of 50 includes too many low-probability tokens, so narrowing to 20 yields cleaner samples without sacrificing diversity.

In [ ]:
# Load best weights for inference
projection_head.load_state_dict(torch.load('/kaggle/working/best_projection_head.pth', weights_only=True))
decoder.load_state_dict(torch.load('/kaggle/working/best_decoder.pth', weights_only=True))
projection_head.eval()
decoder.eval()


def caption_generator(image_feature, max_len=1000):
    full_stop_token = 764          # GPT-2 token for '.'
    generated = [BOS_ID]

    with torch.no_grad():
        image_batch = image_feature.unsqueeze(0).to(device)  # (1, 64, 768)
        memory      = projection_head(image_batch)            # (1, 64, 512)

        for _ in range(max_len):
            caption_tensor = torch.tensor(generated).unsqueeze(0).to(device)
            logits         = decoder(memory, caption_tensor)   # (1, 50257, T)

            top_vals, top_idx = torch.topk(logits[0, :, -1], 50)
            probs      = torch.softmax(top_vals, dim=-1)
            choice     = torch.multinomial(probs, 1, replacement=True).item()
            next_token = top_idx[choice].item()

            if next_token == EOS_ID or next_token == 0:
                break
            generated.append(next_token)
            if next_token == full_stop_token:
                break
            if len(generated) > 3 and generated[-1] == generated[-2] == generated[-3]:
                break

    return tokenizer.decode(generated[1:]).strip()


# Qualitative check — 5 random validation images
sample_keys = random.sample(val_keys, 5)
for key in sample_keys:
    image_feature = torch.from_numpy(get_embedding(key).copy()).float()
    predicted     = caption_generator(image_feature)
    true_captions = list(flickr_captions_dict[key].values())
    print(f'Image: {key}')
    print(f'True:  {true_captions[0]}')
    print(f'Pred:  {predicted}')
    print()

## 12. BLEU-4 Evaluation

All 5 reference captions per image are used for each BLEU computation (multi-reference BLEU). `SmoothingFunction().method4` handles the zero n-gram count problem that occurs with short or imprecise predictions.

Evaluated on the full validation set (3,179 images × 5 references).

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()


smoothie = SmoothingFunction().method4
bleu_scores = []

for key in val_keys:
    image_feature = torch.from_numpy(get_embedding(key).copy()).float()
    predicted     = caption_generator(image_feature)

    references  = [clean_text(cap).split() for cap in flickr_captions_dict[key].values()]
    hypothesis  = clean_text(predicted).split()

    score = sentence_bleu(
        references, hypothesis,
        smoothing_function=smoothie,
        weights=(0.25, 0.25, 0.25, 0.25)
    )
    bleu_scores.append(score)

avg_bleu = sum(bleu_scores) / len(bleu_scores)
print(f'Average BLEU-4 on {len(val_keys)} validation images: {avg_bleu:.4f}')
# OUT: Average BLEU Score: 0.0674

## Results

**BLEU-4: 0.0674** — a 2.6× improvement over v2 (0.026).

Qualitative observations:
- The model generates **grammatically fluent** sentences — switching to a Transformer decoder and a descriptive dataset (Flickr30k) fixed the hallucination / repetition collapse seen in v1/v2
- Captions describe **generic scenes** well: "A dog is running in a field", "Two people are sitting on a bench"
- **Visual grounding is weak**: specific objects, colors, and relationships are often wrong
- The model sometimes ignores the image entirely and generates a plausible-but-generic caption — the ProjectionHead + cross-attention connection is too shallow to enforce tight image-text binding

Sample predictions:

| True caption | Predicted caption |
|---|---|
| A man in a red shirt climbing a rock face | A man is standing on a mountain |
| Children playing in a fountain on a hot day | A group of people are walking in a park |
| A dog catches a frisbee in mid-air | A dog is running in a field |
| Two men in suits are shaking hands | Two men are standing next to each other |

## What This Version Revealed

### What worked
- **Multi-scale ConvNeXt → 64 spatial tokens:** giving the decoder a sequence of image patches (rather than one global vector) lets cross-attention select relevant regions per decoding step. This is why BLEU jumped from 0.026 to 0.0674.
- **LMDB pre-computation:** decoupling feature extraction from training sped up iteration dramatically — epoch time dropped from hours to minutes.
- **GPT-2 tokenizer:** BPE handles rare words and names cleanly; no OOV tokens polluting the vocabulary distribution.

### What didn't work — and why v3b changed it

1. **Custom Transformer vs GPT-2:** the decoder is trained from scratch with random initialization. GPT-2 already knows language — fine-tuning it should give far better fluency than training a 6-layer decoder from zero.

2. **Weak image-text coupling via ProjectionHead:** a two-layer linear head is a thin bottleneck between rich visual features and the decoder's cross-attention. v3b addresses this with a Perceiver that compresses (64, 768) → (16, 768) latents, and injects them at **every** GPT-2 layer via CrossAttentionBlock — tighter, more pervasive conditioning.

3. **Top-k=50 + multinomial without temperature tuning:** sampling is stochastic; the same image can generate different captions across runs. Temperature=0.7 (added in Phase 2d) makes sampling more deterministic and consistent.

## Architecture Comparison Table

| Version | Notebook | Framework | Dataset | Image Encoder | Decoder | BLEU-4 |
|---|---|---|---|---|---|---|
| v0 (Colab) | — | TF/Keras | Instagram | InceptionV3 `layers[-2]` flat 2048 | Embedding + LSTM(256) | ~0 (crash) |
| v1 | `v1_inceptionv3_lstm` | TF/Keras | Instagram | InceptionV3 `layers[-2]` flat 2048 | Embedding + LSTM(512), Add/Concat | ~0 (collapse) |
| v2 | `v2_inceptionv3_attention` | TF/Keras | Instagram | InceptionV3 `mixed10` spatial (64×2048) | Embedding + TemporalImageAttention + LSTM(256) | 0.026 |
| **v3a** (this) | `v3a_convnext_transformer` | PyTorch | Flickr30k | ConvNeXt-Tiny multi-scale (64×768) | ProjectionHead + Custom TransformerDecoder (6L, 8H, 512) | **0.0674** |
| v3b | `v3b_perceiver_gpt2` | PyTorch | Flickr30k | ConvNeXt-Tiny multi-scale (64×768) | Perceiver(16 latents) + GPT-2 + CrossAttention(×12) | 0.0656 |
| v3c | `v3c_coco_finetuning` | PyTorch | COCO 2017 | ConvNeXt-Tiny multi-scale (64×768) | Perceiver + GPT-2 + CrossAttention(×12), COCO fine-tuned | **0.1341** |